# 14 - Management hierarchy screen

This notebook presents the bounded fixed-model comparison of `management`, `scheme_management` and `management_group`. It performs no model refits: run `../scripts/run_management_screen.py` first to recreate the ignored runtime evidence. The labelled local test, competition predictions and oversampling artefacts remain outside this analysis.

In [1]:
from pathlib import Path

import pandas as pd

STAGE_DIR = Path.cwd().parent
PROJECT_DIR = STAGE_DIR.parent
RUNTIME_DIR = PROJECT_DIR / '.runtime' / 'management-screen'
DATA_DIR = STAGE_DIR / 'data'
required = [RUNTIME_DIR / 'policy-register.csv', RUNTIME_DIR / 'frozen-summary.csv']
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f'Run the management screen first; missing={missing!r}')
print(f'Runtime evidence: {RUNTIME_DIR}')

Runtime evidence: C:\_Source\Imperial-ML-AI-Live\Capstone\imperial-capstone\.runtime\management-screen


## Audit boundary

`management` is complete and deterministically maps to five management groups. `scheme_management` is a parallel, incomplete source rather than a hierarchy level. Conservative normalisation changes case and repeated whitespace only; no fuzzy matching or target-informed encoding is used.

In [2]:
training = pd.read_csv(DATA_DIR / 'TrainingSetValues.csv')
competition = pd.read_csv(DATA_DIR / 'TestSetValues.csv')

def normalise(values):
    return (
        values.astype('string').str.strip().str.replace(r'\s+', ' ', regex=True)
        .str.casefold().fillna('__missing__')
    )

audit_rows = []
for frame_name, frame in (('labelled', training), ('competition', competition)):
    management = normalise(frame['management'])
    scheme = normalise(frame['scheme_management'])
    relationship = pd.Series('different', index=frame.index)
    relationship.loc[scheme.eq('__missing__')] = 'scheme_missing'
    relationship.loc[management.eq(scheme)] = 'same'
    audit_rows.append({
        'frame': frame_name,
        'management_levels': management.nunique(),
        'group_levels': normalise(frame['management_group']).nunique(),
        'scheme_levels': scheme.nunique(),
        'same': int(relationship.eq('same').sum()),
        'different': int(relationship.eq('different').sum()),
        'scheme_missing': int(relationship.eq('scheme_missing').sum()),
        'observed_pairs': len(set(zip(management, scheme))),
    })
audit = pd.DataFrame(audit_rows).set_index('frame')
display(audit)
print(
    'Maximum management groups per management level: ',
    training.groupby('management')['management_group'].nunique().max(),
)

,management_levels,group_levels,scheme_levels,same,different,scheme_missing,observed_pairs
frame,,,,,,,
labelled,12,5,12,49336,6186,3878,94
competition,12,5,12,12309,1572,969,66


Maximum management groups per management level:  1


## Predeclared representations

The screen compares each source and group alone, the useful two-field combinations, all three fields, a normalised joint pair and a three-state agreement relationship. Every category encoder remains fold-fitted.

In [3]:
policies = pd.read_csv(RUNTIME_DIR / 'policy-register.csv').set_index('policy')
policies

,label,engineered_features,removed_features,add_management_group,add_composite,add_relationship,rationale
policy,,,,,,,
baseline,Management plus scheme management,29,NaN,False,False,False,Retain the accepted two-source management repr...
management_only,Management only,28,scheme_management,False,False,False,Remove the incomplete parallel scheme field
scheme_only,Scheme management only,28,management,False,False,False,Measure the main management field's independen...
management_group_only,Management group only,28,"management, scheme_management",True,False,False,Replace both source fields with the determinis...
management_plus_group,Management plus group,29,scheme_management,True,False,False,Remove scheme management and expose its coarse...
scheme_plus_group,Scheme management plus group,29,management,True,False,False,Pair the incomplete scheme field with a comple...
all_three,"Management, scheme and group",30,NaN,True,False,False,Test whether the deterministic coarse group he...
composite_only,Management-scheme composite only,28,"management, scheme_management",False,True,False,Replace separate sources with their normalised...
baseline_plus_composite,Baseline plus management-scheme composite,30,NaN,False,True,False,Expose the source interaction while retaining ...


## Frozen-fold screen

Every policy uses the unchanged 55% child-weight-1 depth-8 XGBoost and 45% Random Forest vote. The primary gate requires at least +0.10 percentage points, three fold wins, no fold below -0.25 points and no repair-recall loss beyond two points.

In [4]:
frozen = pd.read_csv(RUNTIME_DIR / 'frozen-summary.csv').set_index('policy')
display_frame = frozen.loc[:, [
    'label', 'mean_accuracy', 'accuracy_change', 'fold_wins',
    'worst_fold_change', 'repair_recall',
    'transformed_features_fold_1', 'passes_gate',
]].copy()
for column in ('mean_accuracy', 'accuracy_change', 'worst_fold_change', 'repair_recall'):
    display_frame[column] = display_frame[column].map(lambda value: f'{value:.3%}')
display_frame

,label,mean_accuracy,accuracy_change,fold_wins,worst_fold_change,repair_recall,transformed_features_fold_1,passes_gate
policy,,,,,,,,
management_only,Management only,81.637%,0.013%,3,-0.126%,34.830%,289,False
baseline,Management plus scheme management,81.625%,0.000%,0,0.000%,34.859%,301,False
management_plus_group,Management plus group,81.610%,-0.015%,2,-0.126%,34.772%,294,False
management_group_only,Management group only,81.593%,-0.032%,2,-0.189%,34.569%,282,False
baseline_plus_composite,Baseline plus management-scheme composite,81.578%,-0.046%,2,-0.210%,34.830%,344,False
composite_only,Management-scheme composite only,81.572%,-0.053%,2,-0.179%,34.975%,320,False
scheme_only,Scheme management only,81.557%,-0.067%,1,-0.189%,34.743%,289,False
scheme_plus_group,Scheme management plus group,81.553%,-0.072%,1,-0.137%,34.714%,294,False
baseline_plus_relationship,Baseline plus relationship state,81.549%,-0.076%,1,-0.189%,34.975%,304,False


## Decision

Retain the accepted `management` plus `scheme_management` policy. Management-only is 0.013 points higher, wins three folds and removes twelve transformed columns, but its gain is far below the promotion threshold. Keep it as a future parsimony candidate under a separately declared non-inferiority rule. Coarse groups, joint pairs and disagreement states do not improve the accepted trees. No primary promotion means no grouped robustness run. Move the next bounded hierarchy loop to extraction type, then source, quality and waterpoint families.